#### Week 8 - Computer Vision: see, detect, and map the built environment

Last week (Week 7) we **named typologies and trained a classifier** on whole images.
This week we (1) look *inside* the image the way a vision model does, (2) **detect and
count objects** in each photo, and (3) **map** those results across the site.

Flow: recap Week 7 (with a backup) -> classify the whole picture -> detect objects ->
put it all on an interactive map.

##### > Two ready-made cases (and how to use your own)

This week ships with **two** geotagged cases so you can see the whole pipeline:
- **detroit** - the Week-7 case (Detroit). Coordinates live in the *filename*.
- **gainesville** - street-level UF photos (`moving` / `still`) with **GPS baked into the
  photo (EXIF)**, plus a ready-made detection CSV for Section 2.

Pick a case below. To use **your own** photos, drop them in a folder of subfolders and add
it to `CASES` (phone photos already carry EXIF GPS).

To use GPU:

- Runtime → Change runtime type
- Under Hardware accelerator, pick T4 GPU (not "None"/"CPU")
- Click Save

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU -> Runtime > Change runtime type > GPU, then Runtime > Restart, and re-run.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir("drive/MyDrive/")

In [ ]:
# --- one-time installs (Colab); does nothing if already present ---
import importlib.util, subprocess, sys
for pkg, pip_name in [('open_clip', 'open_clip_torch'), ('ipywidgets', 'ipywidgets'),
                      ('minisom', 'minisom'), ('folium', 'folium')]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

import csv, math, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

# In Colab, mount Drive and point the CASES paths at your shared folder:
# from google.colab import drive; drive.mount('/content/drive')

# ============ PREPARED CASES (edit paths if needed) ===============
CASES = {
    'detroit': 'data_MLA',           # Week-7 case; coordinates are in the FILENAME
    'gainesville': 'data_gainesville',   # UF street photos; coordinates are in EXIF GPS
}
# ===================================================================

MAX_IMAGES = 400        # cap per perspective (speed / memory safety belt)
SEED = 46
IMG_EXTS = {'.jpg', '.jpeg', '.png'}

def list_images(folder):
    return sorted(p for p in Path(folder).glob('*')
                  if p.is_file() and p.suffix.lower() in IMG_EXTS)

def _name_latlon(p):
    # files named 'lat_lon.jpg' -> (lat, lon)
    parts = Path(p).stem.split('_')
    try:
        if len(parts) == 2:
            return (float(parts[0]), float(parts[1]))
    except ValueError:
        pass
    return None

def _exif_latlon(p):
    # GPS baked into the photo (EXIF) -> (lat, lon)
    try:
        ex = Image.open(p)._getexif() or {}
        gps = ex.get(34853)          # GPSInfo tag
        if not gps:
            return None
        to_deg = lambda v: float(v[0]) + float(v[1]) / 60 + float(v[2]) / 3600
        lat, lon = to_deg(gps[2]), to_deg(gps[4])
        if gps.get(1) == 'S': lat = -lat
        if gps.get(3) == 'W': lon = -lon
        return (lat, lon)
    except Exception:
        return None

def get_latlon(p):
    # works for both cases: try the filename first, then EXIF
    return _name_latlon(p) or _exif_latlon(p)

case = widgets.Dropdown(options=list(CASES), value='detroit', description='Case:')
display(case)

#### 0. Pick - a case and a perspective

In [ ]:
# PICK: which case + perspective (subfolder) to work on.
CASE_ROOT = Path(CASES[case.value])
assert CASE_ROOT.exists(), f'Folder not found: {CASE_ROOT.resolve()} -- fix its path in CASES.'
PERSPECTIVES = [d.name for d in sorted(CASE_ROOT.iterdir()) if d.is_dir() and list_images(d)]
assert PERSPECTIVES, f'No image subfolders found under {CASE_ROOT}.'

perspective = widgets.Dropdown(options=PERSPECTIVES, value=PERSPECTIVES[0], description='Perspective:')
print('Case:', case.value, '| perspectives:', PERSPECTIVES)
display(perspective)

In [ ]:
# CLIP-embed the chosen perspective's images (cached per case+perspective).
try:
    _model
except NameError:
    import torch, open_clip
    _device = 'cuda' if torch.cuda.is_available() else 'cpu'
    _model, _, _preprocess = open_clip.create_model_and_transforms(
        'ViT-B-32', pretrained='laion2b_s34b_b79k')
    _model = _model.to(_device).eval()
    print('CLIP loaded on', _device)

_cache = globals().get('_cache', {})
def embed_perspective(root, name):
    key = (str(root), name)
    if key in _cache:
        return _cache[key]
    import torch
    fs = list_images(root / name)[:MAX_IMAGES]
    feats, keep = [], []
    for i in range(0, len(fs), 64):
        ims, ok = [], []
        for p in fs[i:i + 64]:
            try:
                ims.append(_preprocess(Image.open(p).convert('RGB'))); ok.append(p)
            except Exception:
                pass
        if not ims:
            continue
        with torch.no_grad():
            f = _model.encode_image(torch.stack(ims).to(_device))
            f = f / f.norm(dim=-1, keepdim=True)
        feats.append(f.cpu().numpy()); keep += ok
    Xe = np.concatenate(feats).astype(np.float32)
    _cache[key] = (keep, Xe)
    return keep, Xe

files, X = embed_perspective(CASE_ROOT, perspective.value)
ll = [get_latlon(p) for p in files]
print(f'{perspective.value}: {len(files)} images embedded -> {X.shape[1]}-d features; '
      f'{sum(1 for c in ll if c)} geotagged')

#### Recap Week 7 - your labels (with a backup)

We need last week's **labels** to keep going. This cell loads your Week-7 `labels.csv`
if it is there; if not, it **rebuilds** the typologies from scratch (SOM -> group ->
auto-name) so you are never stuck. Either way you leave this cell with a label per image.

In [ ]:
# Load Week-7 labels if present; otherwise regenerate them (backup).
label_csv = CASE_ROOT / perspective.value / 'labels.csv'
y = None
if label_csv.exists():
    by_name = {r['filename']: r['label'] for r in csv.DictReader(open(label_csv, encoding='utf-8'))}
    got = [by_name.get(Path(p).name) for p in files]
    if all(g is not None for g in got):
        y = np.array(got)
        print('loaded labels from', label_csv)

if y is None:
    print('no complete labels.csv -- rebuilding typologies from scratch (backup)')
    from minisom import MiniSom
    from sklearn.cluster import KMeans
    m = max(3, min(8, int(round((len(X) / 6) ** 0.5))))
    som = MiniSom(m, m, X.shape[1], sigma=1.0, learning_rate=0.5,
                  activation_distance='cosine', random_seed=SEED)
    som.random_weights_init(X); som.train_random(X, 1500)
    win = np.array([som.winner(x) for x in X])
    Wflat = som.get_weights().reshape(m * m, X.shape[1])
    K = max(2, min(6, m))
    node_group = KMeans(K, n_init=10, random_state=SEED).fit_predict(Wflat)
    clust = node_group[win[:, 0] * m + win[:, 1]]
    y = np.array([f'type_{c}' for c in clust])

classes, counts = np.unique(y, return_counts=True)
print('classes:', dict(zip(classes.tolist(), counts.tolist())))

### Section 1 - Classify the whole picture

A classic computer-vision task: show the model an image, it returns **one label** for the
whole thing. Last week's classifier already does this on top of **CLIP** - a large vision
model that was *pretrained* on millions of images. Reusing it is called **transfer
learning**. Below we (a) run that whole-image classifier, (b) train a small CNN of your
own, and (c) look *inside* that trained CNN to see which pixels it leaned on.

In [ ]:
# (a) Whole-image classification: train a light head on the CLIP features, read a score.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

_, cnts = np.unique(y, return_counts=True)
strat = y if cnts.min() >= 2 else None
Xtr, Xte, ytr, yte, itr, ite = train_test_split(
    X, y, np.arange(len(y)), test_size=0.25, random_state=SEED, stratify=strat)
clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
print(f'whole-image test accuracy: {clf.score(Xte, yte):.2f}')

# a small gallery of predictions on held-out images
show = ite[:8]; cols = 4; rows = math.ceil(len(show) / cols)
plt.figure(figsize=(2.4 * cols, 2.6 * rows))
for j, gi in enumerate(show):
    plt.subplot(rows, cols, j + 1)
    plt.imshow(Image.open(files[gi]).convert('RGB')); plt.axis('off')
    plt.title(f'pred {clf.predict(X[gi:gi+1])[0]}', fontsize=8)
plt.suptitle('whole-image predictions'); plt.tight_layout(); plt.show()

#### (b) Train your own small CNN

Train a tiny CNN **from the pixels** on your labeled images - the model your slides
describe. It is slower and can **overfit** on small data (a teaching point in itself). We
look *inside* this trained CNN in the next cell, so keep `TRAIN_CNN = True` to see it.

In [ ]:
TRAIN_CNN = True        # train a small CNN on the raw pixels (needed by the Grad-CAM below)

if TRAIN_CNN:
    import torch, torch.nn as nn
    from torchvision import transforms
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder(); yi = torch.tensor(le.fit_transform(y))
    tf2 = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])
    imgs = torch.stack([tf2(Image.open(p).convert('RGB')) for p in files])
    n = len(imgs); idx = torch.randperm(n); cut = int(0.75 * n)
    tr, te = idx[:cut], idx[cut:]
    net = nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(32 * 16 * 16, 64), nn.ReLU(),
        nn.Linear(64, len(le.classes_)))
    opt = torch.optim.Adam(net.parameters(), 1e-3); lossf = nn.CrossEntropyLoss()
    for ep in range(100):
        net.train(); opt.zero_grad()
        loss = lossf(net(imgs[tr]), yi[tr]); loss.backward(); opt.step()
    net.eval()
    acc = (net(imgs[te]).argmax(1) == yi[te]).float().mean().item()
    print(f'tiny-CNN test accuracy: {acc:.2f}  (compare with the CLIP head above)')
else:
    print('TRAIN_CNN is off -- skipping (the Grad-CAM below needs it).')

#### (c) How your CNN "sees" - a look inside

A CNN builds its answer from **local patterns** (edges -> textures -> parts). Here we look
inside **the CNN you just trained**: we run it on one held-out image and draw a **heat map**
(Grad-CAM) of the pixels it leaned on to pick its class - the intuition from your slides,
made visible on your own model.

In [ ]:
# Grad-CAM on YOUR trained CNN: which pixels drove its decision?
import torch
assert 'net' in globals(), 'Train the CNN first: set TRAIN_CNN = True in the cell above and run it.'

net.eval()
last_conv = net[3]                       # the last Conv2d layer in your small CNN
acts, grads = {}, {}
h1 = last_conv.register_forward_hook(lambda m, i, o: acts.__setitem__('v', o.detach()))
h2 = last_conv.register_full_backward_hook(lambda m, gi, go: grads.__setitem__('v', go[0].detach()))

pick = int(te[0]) if len(te) else 0      # a held-out image
out = net(imgs[pick:pick + 1]); cls = int(out.argmax(1))
net.zero_grad(); out[0, cls].backward()
h1.remove(); h2.remove()

w = grads['v'].mean((2, 3), keepdim=True)                  # importance of each feature map
cam = torch.relu((w * acts['v']).sum(1))[0]                # weighted sum -> heat map
cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-6)

pil = Image.open(files[pick]).convert('RGB')
heat = np.array(Image.fromarray((cam.numpy() * 255).astype('uint8')).resize(pil.size))
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(pil); ax[0].set_title('image'); ax[0].axis('off')
ax[1].imshow(pil); ax[1].imshow(heat, cmap='jet', alpha=0.45)
ax[1].set_title(f'where your CNN looked  (pred: {le.classes_[cls]})'); ax[1].axis('off')
plt.tight_layout(); plt.show()

### Section 2 - Detect objects in each photo

Whole-image classification gives one label per photo. **Object detection** goes further:
it finds and **counts** the things inside each image (cars, trees, windows, people...).
Two ways to do it - pick whichever you like; both make the **same CSV**
(`latitude, longitude, datetime, filename, object_name, object_quantity`).

#### Option A (easiest) - the GulfSouth web tool

No code. Open the tool, drag in your images, download the CSV:
**https://guozifeng91.github.io/GulfSouth/objextract/image_extract.html**
It runs a pretrained detector (SSD MobileNet v2, OpenImages) in your browser and reads the
GPS from each photo's EXIF. *Works with the `gainesville` case and your own phone photos*
(they carry EXIF GPS). Then load the CSV below.

In [ ]:
# Load a CSV produced by the GulfSouth web tool (or any file with the same columns).
# Default: look inside the current case/perspective folder (the gainesville case ships
# with one), then fall back to a file you downloaded next to the notebook.
DETECTION_CSV = CASE_ROOT / perspective.value / 'detection_output.csv'
if not Path(DETECTION_CSV).exists():
    DETECTION_CSV = Path('detection_output.csv')      # <- or point this at your download

det_rows = []
if Path(DETECTION_CSV).exists():
    det_rows = [r for r in csv.DictReader(open(DETECTION_CSV, encoding='utf-8'))
                if r.get('object_name')]
    print(f'loaded {len(det_rows)} detections from {DETECTION_CSV}')
else:
    print('no CSV yet -- run the web tool, or use Option B below.')

#### Option B - detect inside this notebook

Runs a pretrained detector (YOLO) right here and writes the **same CSV**. Use this for the
`detroit` case (coords come from the filename) or when you want everything in one place.

In [ ]:
# In-notebook detection with a pretrained YOLO model -> same CSV schema.
RUN_DETECTOR = True        # flip to True (first run downloads the model)

if RUN_DETECTOR:
    if importlib.util.find_spec('ultralytics') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])
    from ultralytics import YOLO
    yolo = YOLO('yolov8n.pt')
    cols = ['latitude', 'longitude', 'datetime', 'filename', 'object_name', 'object_quantity']
    out = CASE_ROOT / perspective.value / 'detection_output.csv'
    det_rows = []
    for p in files:
        res = yolo(str(p), verbose=False)[0]
        counts = {}
        for c in res.boxes.cls.tolist():
            nm = res.names[int(c)]
            counts[nm] = counts.get(nm, 0) + 1
        geo = get_latlon(p) or ('', '')
        for obj, q in counts.items():
            det_rows.append({'latitude': geo[0], 'longitude': geo[1], 'datetime': '',
                             'filename': Path(p).name, 'object_name': obj, 'object_quantity': q})
    with open(out, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(det_rows)
    print(f'wrote {len(det_rows)} detections -> {out}')
else:
    print('RUN_DETECTOR is off -- using the CSV from Option A (if loaded).')

### Section 3 - Put it on the map

Every result carries a **lat/lon**, so we can place it geographically - an automated
**site inventory**. Below: an interactive map of the detected objects, colored by type,
with a density heat layer.

In [ ]:
# Interactive map (folium): detected objects, colored by type, plus a density heatmap.
import folium
from folium.plugins import HeatMap

pts = []
for r in det_rows:
    try:
        pts.append((float(r['latitude']), float(r['longitude']),
                    r['object_name'], int(float(r['object_quantity']))))
    except (ValueError, TypeError):
        pass
assert pts, 'No geotagged detections to map -- load a CSV (Option A) or run Option B.'

clat = np.mean([p[0] for p in pts]); clon = np.mean([p[1] for p in pts])
palette = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue', 'black']
kinds = sorted({p[2] for p in pts})
color = {k: palette[i % len(palette)] for i, k in enumerate(kinds)}

fmap = folium.Map(location=[clat, clon], zoom_start=16)
for lat, lon, obj, q in pts:
    folium.CircleMarker([lat, lon], radius=3 + q, color=color[obj], fill=True,
                        fill_opacity=0.7, popup=f'{obj} x{q}').add_to(fmap)
HeatMap([[lat, lon] for lat, lon, _, _ in pts]).add_to(
    folium.FeatureGroup(name='density').add_to(fmap))
folium.LayerControl().add_to(fmap)
fmap

#### (Advanced) ArcGIS Online

Prefer ArcGIS? Export a clean CSV and upload it to ArcGIS Online (Map Viewer -> Add ->
From file), or use the `arcgis` Python API. The CSV already has `latitude`/`longitude`,
so ArcGIS maps it directly.

In [ ]:
# Export the detections for ArcGIS Online upload.
arc_out = 'detections_for_arcgis.csv'
cols = ['latitude', 'longitude', 'datetime', 'filename', 'object_name', 'object_quantity']
with open(arc_out, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=cols); w.writeheader()
    for r in det_rows:
        w.writerow({k: r.get(k, '') for k in cols})
print('wrote', arc_out, '-- upload this to ArcGIS Online, or feed the arcgis Python API.')

You just ran a full **computer-vision** pass on the same project: classified whole images,
looked *inside* a CNN, **detected and counted** objects, and mapped them into a site
inventory. Next: the **generative** week, where we turn these readings into new design
variations.